In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
raw_dir = '../data/raw'

In [3]:
movies = pd.read_csv(f'{raw_dir}/movies.csv')
genome_scores = pd.read_csv(f'{raw_dir}/genome-scores.csv')

In [4]:
dna_matrix = genome_scores.pivot(index='movieId', columns='tagId', values='relevance')

# Not all movies have genome scores. We filter our movies df to only include the ones that do.
valid_movie_ids = dna_matrix.index
content_df = movies[movies['movieId'].isin(valid_movie_ids)].copy()

In [5]:
def get_content_recommendations(movie_title, top_n=5):
    """Returns top N similar movies based purely on Tag Genome DNA."""
    
    # 1. Find the exact movieId using string containment
    idx_search = content_df[content_df['title'].str.contains(movie_title, case=False, na=False)]
    
    if idx_search.empty:
        return f"Movie '{movie_title}' not found in the Genome database."
    
    # Extract the ID of the first match
    target_movie_id = idx_search.iloc[0]['movieId']
    exact_title = idx_search.iloc[0]['title']
    print(f"Analyzing DNA and routing recommendations for: {exact_title}\n")
    
    # 2. Extract the DNA vector for the target movie
    target_dna = dna_matrix.loc[target_movie_id].values.reshape(1, -1)
    
    # 3. Calculate cosine similarity against the entire DNA matrix
    # This mathematically finds the vectors pointing in the exact same direction
    sim_scores = cosine_similarity(target_dna, dna_matrix).flatten()
    
    # 4. Sort and get top N matches (excluding the movie itself, which is always 1.0)
    # argsort() sorts ascending, so we take the end of the array and reverse it
    similar_indices = sim_scores.argsort()[-(top_n+1):-1][::-1]
    
    # 5. Map the indices back to actual movie IDs and Titles
    top_movie_ids = dna_matrix.index[similar_indices]
    
    results = content_df[content_df['movieId'].isin(top_movie_ids)].copy()
    
    # Add the similarity score for transparency
    # We create a temporary mapping dataframe to ensure scores align perfectly with the IDs
    score_mapping = pd.DataFrame({
        'movieId': top_movie_ids,
        'similarity_score': sim_scores[similar_indices]
    })
    
    results = results.merge(score_mapping, on='movieId')
    results = results.sort_values('similarity_score', ascending=False)
    
    return results[['title', 'genres', 'similarity_score']]

In [6]:
test_movie = "Matrix, The"
recommendations = get_content_recommendations(test_movie)

display(recommendations)

Analyzing DNA and routing recommendations for: Matrix, The (1999)



,title,genres,similarity_score
4,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.898807
3,Equilibrium (2002),Action|Sci-Fi|Thriller,0.897470
2,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.892831
1,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.890542
0,Blade Runner (1982),Action|Sci-Fi|Thriller,0.886513


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import time
import numpy as np

# ==========================================
# 1. PREP THE DNA FOR THE NEURAL NETWORK
# ==========================================
print("Aligning Tag Genome (DNA) with Neural Network Indices...")

# We need a fast lookup table on the GPU so the model can grab DNA during training
dna_dim = 1128
dna_feature_array = np.zeros((num_items, dna_dim))

# Map the pre-calculated genome scores to our continuous PyTorch item indices
for movie_id in dna_matrix.index:
    # Only grab DNA for movies that survived our earlier 500-rating downsampling
    if movie_id in item_encoder.classes_:
        item_idx = item_encoder.transform([movie_id])[0]
        dna_feature_array[item_idx] = dna_matrix.loc[movie_id].values

# Move the whole DNA matrix to the GPU once to save Dataloader overhead
dna_tensor = torch.FloatTensor(dna_feature_array).to(device)
print("DNA Tensor ready and loaded to GPU!")

# ==========================================
# 2. THE ULTIMATE HYBRID ARCHITECTURE
# ==========================================
class UltimateHybridNeuMF(nn.Module):
    def __init__(self, num_users, num_items, dna_tensor, embed_size=64):
        super(UltimateHybridNeuMF, self).__init__()
        
        # Store the DNA lookup table
        self.dna_tensor = dna_tensor
        
        # Embeddings
        self.user_embed_gmf = nn.Embedding(num_users, embed_size)
        self.item_embed_gmf = nn.Embedding(num_items, embed_size)
        self.user_embed_mlp = nn.Embedding(num_users, embed_size)
        self.item_embed_mlp = nn.Embedding(num_items, embed_size)
        
        # --- TWEAK 4: HYBRID INJECTION ---
        # Compress the massive 1128-feature DNA vector down to 64 dense features
        self.dna_compress = nn.Linear(1128, 64)
        
        # --- TWEAK 2: BATCH NORMALIZATION ---
        # MLP Layers now take User(64) + Item(64) + DNA(64) = 192 inputs
        self.fc1 = nn.Linear(192, 128)
        self.bn1 = nn.BatchNorm1d(128) # Normalizes data flowing into next layer
        
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)

        # Output Layer (GMF(64) + MLP(32) = 96)
        self.output = nn.Linear(embed_size + 32, 1)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid() 

    def forward(self, user_indices, item_indices):
        # 1. GMF Path (Math)
        user_gmf = self.user_embed_gmf(user_indices)
        item_gmf = self.item_embed_gmf(item_indices)
        gmf_vector = torch.mul(user_gmf, item_gmf)
        
        # 2. Fetch and Compress DNA
        raw_dna = self.dna_tensor[item_indices]
        compressed_dna = self.relu(self.dna_compress(raw_dna))
        
        # 3. MLP Path (Deep Learning + DNA)
        user_mlp = self.user_embed_mlp(user_indices)
        item_mlp = self.item_embed_mlp(item_indices)
        
        # Inject the DNA right alongside the User and Item
        mlp_vector = torch.cat([user_mlp, item_mlp, compressed_dna], dim=1)
        
        # Pass through layers with Batch Normalization
        mlp_vector = self.fc1(mlp_vector)
        mlp_vector = self.bn1(mlp_vector)
        mlp_vector = self.relu(mlp_vector)
        mlp_vector = self.dropout(mlp_vector)
        
        mlp_vector = self.fc2(mlp_vector)
        mlp_vector = self.bn2(mlp_vector)
        mlp_vector = self.relu(mlp_vector)
        mlp_vector = self.dropout(mlp_vector)
        
        mlp_vector = self.fc3(mlp_vector)
        mlp_vector = self.bn3(mlp_vector)
        mlp_vector = self.relu(mlp_vector)

        # 4. Combine and Predict
        combined_vector = torch.cat([gmf_vector, mlp_vector], dim=1)
        prediction = self.output(combined_vector)
        
        # Bound output to 0.5 - 5.0
        scaled_prediction = (self.sigmoid(prediction) * 4.5) + 0.5
        
        return scaled_prediction.squeeze()

# ==========================================
# 3. TUNED TRAINING LOOP
# ==========================================
model = UltimateHybridNeuMF(num_users, num_items, dna_tensor).to(device)

# --- TWEAK 1: HUBER LOSS ---
# SmoothL1Loss acts like MSE for small errors, but ignores massive outliers
criterion = nn.SmoothL1Loss() 

optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-5) # Slightly higher LR due to BatchNorm
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)

epochs = 8
print(f"Starting Ultimate Hybrid Training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        
        optimizer.zero_grad()
        predictions = model(users, items)
        
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    scheduler.step(avg_train_loss)
    
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train Loss (Huber): {avg_train_loss:.4f} | Time: {epoch_time:.2f}s")

Aligning Tag Genome (DNA) with Neural Network Indices...


NameError: name 'num_items' is not defined